# C-tool (Colab)

URL → whisper-jax + pyannote → JSON. Không Gradio (tránh lỗi FastAPI/Pydantic).

1. GPU T4
2. Cell **Reset + cài** (một lần, hoặc mỗi khi có commit mới)
3. Cell **Chạy** — dán URL, bấm Run

In [ ]:
import os, shutil
from pathlib import Path

os.chdir("/content")
%cd /content

for p in ("/content/c-tools", "/content/ctool-venv"):
    path = Path(p)
    if path.exists():
        shutil.rmtree(path)
        print("Deleted", p)

!git clone --depth 1 https://github.com/vcstack/c-tools.git /content/c-tools
!apt-get -y install -qq ffmpeg
%cd /content/c-tools
!git log -1 --oneline
!bash /content/c-tools/colab_setup.sh

In [ ]:
# @title Chạy C-tool
media_url = ""  # @param {type:"string"}
hf_token = ""  # @param {type:"string"}
model = "large-v2"  # @param ["tiny", "base", "small", "medium", "large-v2", "large-v3"]
language = ""  # @param {type:"string"}
min_speakers = 1  # @param {type:"integer"}
max_speakers = 10  # @param {type:"integer"}

import json, os, subprocess
from pathlib import Path
from google.colab import files

os.chdir("/content")
ROOT = Path("/content/c-tools")
PY = Path("/content/ctool-venv/bin/python")
if not PY.is_file():
    raise SystemExit("Chưa có venv. Chạy cell Reset + cài trước.")
if not (media_url or "").strip():
    raise SystemExit("Dán Media URL (YouTube, ...).")

os.chdir(ROOT)
out = ROOT / "output" / "result.json"
out.parent.mkdir(exist_ok=True)
env = os.environ.copy()
env["HF_TOKEN"] = (hf_token or "").strip()
env["HUGGING_FACE_HUB_TOKEN"] = env["HF_TOKEN"]
env["MPLBACKEND"] = "Agg"

cmd = [
    str(PY), "main.py",
    "--input", media_url.strip(),
    "--output", str(out),
    "--model", model,
    "--min-speakers", str(int(min_speakers)),
    "--max-speakers", str(int(max_speakers)),
    "--device", "auto",
]
if (language or "").strip():
    cmd.extend(["--language", language.strip()])

print(" ".join(cmd))
code = subprocess.call(cmd, env=env, cwd=str(ROOT))
if code != 0 or not out.is_file():
    raise SystemExit(f"Pipeline failed (exit {code})")

data = json.loads(out.read_text(encoding="utf-8"))
print("file:", data["source"]["filename"], "duration:", data["source"]["duration"])
print("speakers:", data["speakers"], "segments:", len(data["segments"]))
for seg in data["segments"][:30]:
    print(f'{seg["start"]:7.2f} → {seg["end"]:7.2f}  {seg["speaker"]}  {seg["text"]}')
files.download(str(out))